# Lane 3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/qwen-symbolic-thinking/blob/main/notebooks/lane3.ipynb)

A card that works through a queue of runs kept on the Hub (`runs/queue_lane3.json` in the project repo): fine-tunes, each followed by its screen, and synthetic-row generation. Each finished run leaves `runs/done/<name>.json`; new runs can be added to the queue while the card works, and the cell ends after 45 idle minutes.

Runtime: a GPU (the RTX PRO 6000 or an A100 fits the 9B in bf16 at batch 8) and the `HF_TOKEN` secret with write access to the project repo. Run C1, restart the session (Runtime > Restart session), then run C2.


In [ ]:
# C1: install, then Runtime > Restart session
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("git+https://github.com/AbstractEyes/qwen-symbolic-thinking@v0.1.13", "transformers>=5.0", "accelerate>=1.0", "peft>=0.18")

FAST_PATH = True   # the linear-attention kernels Qwen3.5 uses; causal-conv1d compiles for about 5 minutes; all or nothing
if FAST_PATH:
    if all(importlib.util.find_spec(m) for m in ("causal_conv1d", "fla", "triton")):
        print("fast path already present")
    else:
        try:
            pip("causal-conv1d", "--no-build-isolation")
            pip("flash-linear-attention", "triton")
            print("fast path on")
        except subprocess.CalledProcessError as e:
            print("fast path off (training still runs, slower):", e)
            subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "causal-conv1d"])

try:                # Colab's torchao 0.10 breaks PEFT when it builds a LoRA (PEFT wants 0.16 or newer, or none at all)
    import torchao
    _v = tuple(int(x) for x in torchao.__version__.split(".")[:2])
    if _v < (0, 16):
        subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
        print("torchao", torchao.__version__, "removed: PEFT cannot build a LoRA beside it")
    else:
        print("torchao", torchao.__version__, "stays")
except ImportError:
    print("no torchao")

print("installed; now Runtime > Restart session, then run C2")

In [ ]:
# C2: work through the lane's queue (runs/queue_lane3.json on the Hub; new runs can be added while this runs)
from qwen_symbolic_thinking.queue import run_queue

ran = run_queue("lane3")
print("ran:", ran)
